# Q&A Loop and Agent Class

This notebook turns your chapter notes into a study-friendly format with explanation-first structure, logical diagrams, inline code comments, and possible output comments inside code cells. [cite:322]


## Lesson goal

In the previous lesson, the **agentic loop** handled the inner tool-calling process until the agent had enough information. In this lesson, we add an **outer Q&A loop** for user interaction and then refactor the logic into an `Agent` class.


## Big idea

This chapter introduces a **two-loop structure**:
- the outer loop handles conversation with the user,
- the inner loop handles tool calling and reasoning.

Then the lesson wraps that behavior into a reusable `Agent` class so the code is easier to reuse, test, and maintain.


## Two-loop diagram

```text
OUTER LOOP (Q&A loop)
  Ask user for input
  |
  +--> if user says 'stop' -> exit program
  |
  +--> otherwise append user message to history
             |
             v
      INNER LOOP (tool-call loop)
        Call model
        |
        +--> if model requests tool(s)
        |       Python executes them
        |       Append tool results
        |       Call model again
        |
        +--> if no more tool calls
                print final assistant reply
                return to OUTER LOOP
```

So the outer loop handles **multiple user turns**, while the inner loop handles **multiple tool-use iterations inside one user turn**. [cite:322]


In [12]:
# Install packages if needed.
# Run this once in a fresh environment.
%pip install -q openai gitsource minsearch pydantic

# Possible output:
# - Installation logs
# - Or a message saying packages are already installed


/workspaces/ai-engineering-buildcamp/.venv/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


## Imports and setup

We use the same building blocks as before: OpenAI client, docs reader, chunking helper, and searchable index.


In [13]:
from openai import OpenAI
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex
import json
from pprint import pprint

openai_client = OpenAI()

# Possible output:
# - Usually no printed output if imports succeed


## Build the documentation index

The notebook needs a knowledge base so the tools have something real to search and update.


In [14]:
reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()
parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print('Files loaded:', len(files))
print('Chunks created:', len(chunked_docs))

# Possible output:
# Files loaded: <some positive integer>
# Chunks created: <some larger positive integer>


Files loaded: 95
Chunks created: 382


## Define the tools

This chapter assumes the same `search` and `add_entry` tools from the previous lesson.


In [15]:
def search(query):
    results = index.search(query=query, num_results=5)
    return results

search_tool = {
    'type': 'function',
    'name': 'search',
    'description': 'Search the documentation database for relevant results based on a query string.',
    'parameters': {
        'type': 'object',
        'properties': {
            'query': {
                'type': 'string',
                'description': 'The search query to look up in the index'
            }
        },
        'required': ['query']
    }
}

def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return 'OK'

add_entry_tool = {
    'type': 'function',
    'name': 'add_entry',
    'description': 'Add a new documentation entry to the index.',
    'parameters': {
        'type': 'object',
        'properties': {
            'filename': {'type': 'string', 'description': 'The source filename associated with the entry'},
            'title': {'type': 'string', 'description': 'The title of the documentation entry'},
            'description': {'type': 'string', 'description': 'A short description summarizing the entry'},
            'content': {'type': 'string', 'description': 'The full content of the documentation entry'},
        },
        'required': ['filename', 'title', 'description', 'content']
    }
}

tools = [search_tool, add_entry_tool]

# Possible output:
# - No printed output
# - The notebook now has two tool schemas and two real Python functions


## Instructions

We continue using a documentation-assistant style system prompt.


In [16]:
instructions = '''
You're a documentation assistant.

Answer the user question using the documentation knowledge base.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.
'''.strip()

# Possible output:
# - No printed output


In [17]:
import json

def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    elif name == 'add_entry':
        result = add_entry(**arguments)
    else:
        result = f'not found tool "{name}"'

    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

# Possible output:
# - No printed output
# - After running this cell, make_call will be available to the loop

## Outer loop with inner tool-call loop

This is the first big step of the chapter. We keep the earlier tool-call loop, but now place it inside a larger loop that repeatedly accepts user input.


In [18]:
tools = [search_tool, add_entry_tool]

message_history = [
    {'role': 'system', 'content': instructions},
]

iteration_number = 1

# Q&A loop
while True:
    user_prompt = input('You:')
    if user_prompt.lower().strip() == 'stop':
        break

    message_history.append({'role': 'user', 'content': user_prompt})

    # tool-call loop
    while True:
        response = openai_client.responses.create(
            model='gpt-4o-mini',
            input=message_history,
            tools=tools,
        )

        print(f'iteraration number {iteration_number}...')
        message_history.extend(response.output)

        has_function_calls = False

        for message in response.output:
            if message.type == 'function_call':
                print(f'executing {message.name}({message.arguments})...')
                tool_call_output = make_call(message)
                message_history.append(tool_call_output)
                has_function_calls = True

            if message.type == 'message':
                text = message.content[0].text
                print('ASSISTANT:', text)

        iteration_number = iteration_number + 1
        print()

        if not has_function_calls:
            break

# Before this block:
# - message_history starts with only the system message.
#
# At this block:
# - Outer loop asks the user for a new question.
# - Inner loop handles all tool calls needed for that one question.
#
# After this block:
# - Control returns to the outer loop for the next user question.
# - Conversation history keeps growing across turns.
#
# Possible output:
# You: how do I create a dashboard?
# iteraration number 1...
# executing search({"query": "create dashboard"})...
# iteraration number 2...
# ASSISTANT: Here is how to create a dashboard...
#
# You: add this answer to the database
# iteraration number 3...
# executing add_entry({...})...
# ASSISTANT: The content has been added to the database.
#
# You: stop


## What each loop does

The **outer Q&A loop** handles repeated user interaction. The **inner tool-call loop** handles repeated tool usage for one user message until the model is done.


```text
Outer loop responsibility:
- ask for user input
- stop if user types 'stop'
- append the user's message
- call the inner loop

Inner loop responsibility:
- call the model
- inspect returned objects
- execute requested tools
- append tool results
- stop when there are no more tool calls
```


## Why refactor into a class

The chapter then points out several problems with the plain script version:
- too much global state,
- hard to reuse,
- hard to test,
- difficult to create multiple agents with different settings.

A class solves this by grouping the agent's data and behavior into one reusable object.


## Agent class diagram

```text
Agent
├── __init__
│     stores: llm_client, model, instructions, tools
├── make_call
│     executes requested tools
├── loop
│     runs the inner tool-call loop for one user prompt
└── qna
      runs the outer Q&A loop for many user prompts
```


## Create the Agent class

This refactor wraps everything into a reusable object. Notice that the inner tool-call loop becomes the `loop(...)` method, while the outer Q&A loop becomes the `qna()` method.


In [19]:
class Agent:

    def __init__(self, llm_client, model, instructions, tools):
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.tools = tools

    def make_call(self, tool_call):
        arguments = json.loads(tool_call.arguments)
        name = tool_call.name

        if name == 'search':
            result = search(**arguments)
        elif name == 'add_entry':
            result = add_entry(**arguments)
        else:
            result = 'not found tool "{name}"'

        return {
            'type': 'function_call_output',
            'call_id': tool_call.call_id,
            'output': json.dumps(result),
        }

    def loop(self, user_prompt, message_history=None):
        if not message_history:
            message_history = [
                {'role': 'system', 'content': self.instructions},
            ]

        message_history.append({'role': 'user', 'content': user_prompt})

        iteration_number = 0

        while True:
            response = self.llm_client.responses.create(
                model=self.model,
                input=message_history,
                tools=self.tools,
            )

            print(f'iteraration number {iteration_number}...')
            message_history.extend(response.output)

            has_function_calls = False

            for message in response.output:
                if message.type == 'function_call':
                    print(f'executing {message.name}({message.arguments})...')
                    tool_call_output = self.make_call(message)
                    message_history.append(tool_call_output)
                    has_function_calls = True

                if message.type == 'message':
                    text = message.content[0].text
                    print('ASSISTANT:', text)

            iteration_number = iteration_number + 1
            print()

            if not has_function_calls:
                break

        return message_history

    def qna(self):
        message_history = [
            {'role': 'system', 'content': instructions},
        ]

        iteration_number = 1

        while True:
            user_prompt = input('You:')
            if user_prompt.lower().strip() == 'stop':
                break

            message_history = self.loop(user_prompt, message_history)

# Before this block:
# - We already know the script version of the loops.
#
# At this block:
# - We move agent behavior into methods.
# - `loop()` handles one user's question internally.
# - `qna()` handles repeated user questions.
#
# After this block:
# - We can create multiple Agent objects with different configs.
#
# Possible output:
# - No printed output when the class is defined


## What each method does

- `__init__`: stores configuration such as client, model, instructions, and tools.
- `make_call`: executes tool requests.
- `loop`: runs the inner tool-call loop for one user message and returns updated history.
- `qna`: runs the outer loop and repeatedly calls `loop`.


## Create an agent instance

Once the class exists, creating a configured agent becomes simple.


In [20]:
agent = Agent(
    llm_client=OpenAI(),
    model='gpt-4o-mini',
    instructions=instructions,
    tools=tools
)

# Possible output:
# - No printed output
# - `agent` now stores the model config and agent behavior together


## Test the inner loop once

The chapter suggests testing `loop(...)` directly with a single question. This is useful because it lets you test one turn without entering the full interactive session.


In [21]:
messages = agent.loop('evidently adshboards')

# Possible output:
# iteraration number 0...
# executing search({"query": "evidently dashboards"})...
# iteraration number 1...
# ASSISTANT: Here is information about Evidently dashboards...
#
# Exact wording and queries may vary


iteraration number 0...
executing search({"query":"evidently dashboards"})...

iteraration number 1...
ASSISTANT: Evidently offers a comprehensive platform designed to support AI workflows, featuring rich dashboards and over 100 built-in evaluations for ML and LLM tasks. Here are key points about Evidently and its dashboards:

1. **Modular Design**: The platform is highly modular, allowing you to start small and expand as needed, without complex setups. You can run local evaluations, share results via a UI, and gradually implement monitoring.

2. **Rich Dashboards**: Evidently provides an intuitive dashboard interface that is faster, cleaner, and more user-friendly, making it easier to visualize metrics and evaluation results.

3. **Built-in Evaluations**: It includes over 100 built-in evaluations to cover various machine learning tasks, ensuring quality testing without needing to implement metrics from scratch.

4. **Collaborative Features**: The platform is built for collaboration, i

## Try the full Q&A experience

Calling `agent.qna()` starts the interactive outer loop.


In [22]:
agent.qna()

# Possible output:
# You: how do I create a dashboard?
# iteraration number 0...
# executing search({...})...
# ASSISTANT: ...
#
# You: add this to the database
# iteraration number 0...
# executing add_entry({...})...
# ASSISTANT: ...
#
# You: stop


iteraration number 0...
executing search({"query":"destroy a dashboard"})...

iteraration number 1...
ASSISTANT: To destroy a dashboard, you can use the following Python API method:

```python
project.dashboard.clear_dashboard()
```

This command will delete all Tabs and Panels on the dashboard. Note that this action does **not** delete the underlying Reports or dataset; it only clears the Panels.



## Why this matters

This chapter effectively builds a small agent framework from scratch:
- an inner tool-call loop,
- an outer Q&A loop,
- and an `Agent` class that packages the behavior.

That is why the lesson says this is the same overall pattern used in production agent frameworks.


## Practice tasks

Try these after studying the notebook:
1. Change `make_call` so unknown tools use an f-string error message
2. Add a `max_iterations` safety limit to `loop()`
3. Pass `message_history` through `qna()` more explicitly
4. Create a second `Agent` instance with different instructions
5. Unit-test `make_call()` separately from the full loop
